# Physics-Aware Optimization of Unified Neural Signed Distance Fields for 3D Indoor Scene Reconstruction
## Google Colab / Kaggle GPU Execution & Reproducibility Pipeline

This notebook provides a staged, reproducible environment on Kaggle / Google Colab with CUDA GPU acceleration for **I²-SDF** and our **physics-aware grounding optimization**.

---
### Staged Execution Workflow Architecture:
- **Section A**: GPU Hardware & Environment Verification
- **Section B**: Repository Setup, Cloning (`https://github.com/Vernit185/i2-sdf`), & File Integrity Check
- **Section C**: Dependency Installation (PyTorch Lightning, Open3D, Trimesh, PyMCubes, OpenCV, etc.)
- **Section D**: Dataset Setup (`bedroom_0` download & camera normalization into `data/synthetic/scan0`)
- **Section E**: Modality & Configuration Verification (OpenEXR, Config Loader, CUDA ops)
- **Section F (Stage 1)**: Sanity Test (`RUN_SANITY_TEST = True` — Fast 2s Unit Tests & Math Verification)
- **Section G (Stage 2)**: Runtime Calibration (`RUN_CALIBRATION = False`, `MAX_STEPS = 1000` — NOT the final experiment)
- **Section H (Stage 3)**: Full Research Experiment (`RUN_FULL_EXPERIMENT = False`, `TRAIN_STEPS = 200000` for Baseline & Physics)
- **Section I (Stage 4)**: Ablation Studies (`RUN_ABLATIONS = False` — Loss Weight Sensitivity)
- **Section J (Stage 5)**: 3D Mesh Extraction, Quantitative Grounding Metrics & Random Forest Analysis
- **Section K**: Persistent Storage & Backup
- **Summary**: Command Reference Cheatsheet


## Section A: GPU Hardware & Environment Verification
Verifies CUDA GPU availability, device name, compute capability, and VRAM memory.

In [ ]:
# ==============================================================================
# SECTION A: GPU HARDWARE VERIFICATION
# ==============================================================================
import os
import sys
import torch

print("=" * 75)
print("               SECTION A: CUDA GPU HARDWARE VERIFICATION")
print("=" * 75)

cuda_available = torch.cuda.is_available()
print(f"[STATUS] torch.cuda.is_available() : {cuda_available}")

if not cuda_available:
    raise RuntimeError(
        "CUDA GPU is NOT available! On Kaggle/Colab, enable a GPU accelerator (T4, P100, or L4/A100) in runtime settings."
    )

gpu_count = torch.cuda.device_count()
gpu_name = torch.cuda.get_device_name(0)
gpu_capability = torch.cuda.get_device_capability(0)
total_memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
allocated_memory_gb = torch.cuda.memory_allocated(0) / (1024 ** 3)
cached_memory_gb = torch.cuda.memory_reserved(0) / (1024 ** 3)

print(f"[STATUS] Detected GPU Count        : {gpu_count}")
print(f"[STATUS] Primary GPU Device Name   : {gpu_name}")
print(f"[STATUS] Compute Capability        : {gpu_capability[0]}.{gpu_capability[1]}")
print(f"[STATUS] Total VRAM Available      : {total_memory_gb:.2f} GB")
print(f"[STATUS] Allocated VRAM            : {allocated_memory_gb:.2f} GB")
print(f"[STATUS] Reserved VRAM             : {cached_memory_gb:.2f} GB")
print(f"[STATUS] PyTorch Version           : {torch.__version__}")
print(f"[STATUS] CUDA Version in PyTorch   : {torch.version.cuda}")
print("=" * 75)
print("\n[nvidia-smi Output]:")
!nvidia-smi


## Section B: Repository Setup & File Verification
Clones `https://github.com/Vernit185/i2-sdf`, navigates into the active `i2-sdf-main` source tree, and checks that all physics-aware modules and test suites exist.

In [ ]:
# ==============================================================================
# SECTION B: REPOSITORY SETUP & FILE VERIFICATION
# ==============================================================================
import os
import sys
import shutil

print("=" * 75)
print("             SECTION B: REPOSITORY SETUP & INTEGRITY CHECK")
print("=" * 75)

REPO_URL = "https://github.com/Vernit185/i2-sdf.git"
TARGET_CLONE_DIR = "/content/i2-sdf" if os.path.exists("/content") else "/kaggle/working/i2-sdf"

# 1. Clone repository if needed
if os.path.exists("main_recon.py") and os.path.exists("model"):
    PROJECT_ROOT = os.path.abspath(".")
    print(f"[INFO] Already within active source directory: {PROJECT_ROOT}")
elif os.path.exists("i2-sdf-main/main_recon.py"):
    PROJECT_ROOT = os.path.abspath("i2-sdf-main")
    os.chdir(PROJECT_ROOT)
    print(f"[INFO] Changed working directory to: {PROJECT_ROOT}")
else:
    if not os.path.exists(TARGET_CLONE_DIR):
        print(f"[INFO] Cloning repository from {REPO_URL} into {TARGET_CLONE_DIR} ...")
        !git clone {REPO_URL} {TARGET_CLONE_DIR}
    
    if os.path.exists(os.path.join(TARGET_CLONE_DIR, "i2-sdf-main", "main_recon.py")):
        PROJECT_ROOT = os.path.join(TARGET_CLONE_DIR, "i2-sdf-main")
    elif os.path.exists(os.path.join(TARGET_CLONE_DIR, "main_recon.py")):
        PROJECT_ROOT = TARGET_CLONE_DIR
    else:
        PROJECT_ROOT = os.path.abspath(".")
    
    os.chdir(PROJECT_ROOT)
    print(f"[INFO] Working directory set to: {PROJECT_ROOT}")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"\n[STATUS] Current Directory: {os.getcwd()}")

# 2. Verify existence of required physics and core files
required_files = [
    "model/physics.py",
    "model/network/__init__.py",
    "model/trainer/recon.py",
    "model/physics_classifier.py",
    "tests/test_physics.py",
    "main_recon.py",
    "environment.yml",
    "config/synthetic.yml",
    "config/synthetic_physics.yml",
    "test_grounding_poc.py",
    "data/normalize_cameras.py"
]

missing_files = []
for rel_path in required_files:
    full_p = os.path.join(PROJECT_ROOT, rel_path)
    exists = os.path.exists(full_p)
    status_str = "EXISTS" if exists else "MISSING"
    print(f"  • {rel_path:<35} : [{status_str}]")
    if not exists:
        missing_files.append(rel_path)

if missing_files:
    raise FileNotFoundError(f"Missing critical project files: {missing_files}")
else:
    print("\n[SUCCESS] All core and physics files verified successfully!")
print("=" * 75)


## Section C: Dependency Installation
Installs pinned runtime dependencies matching `environment.yml` for PyTorch Lightning, Open3D, Trimesh, PyMCubes, and fast-pytorch-kmeans.

In [ ]:
# ==============================================================================
# SECTION C: DEPENDENCY INSTALLATION
# ==============================================================================
import os
import sys

print("=" * 75)
print("             SECTION C: INSTALLING REQUIRED DEPENDENCIES")
print("=" * 75)

# System packages
print("[INFO] Installing system utilities (ffmpeg, libopenexr-dev)...\n")
!apt-get update -qq && apt-get install -y -qq ffmpeg libopenexr-dev > /dev/null 2>&1

# Python packages
print("[INFO] Installing Python packages...\n")
!pip install -q \
    "pytorch-lightning==1.9.0" \
    "torchmetrics==0.11.4" \
    "open3d==0.17.0" \
    "trimesh==3.21.4" \
    "PyMCubes==0.1.4" \
    "fast-pytorch-kmeans==0.1.9" \
    "opencv-python==4.7.0.72" \
    "lpips==0.1.4" \
    "scikit-image==0.20.0" \
    "scikit-learn==1.2.2" \
    "scipy==1.9.1" \
    "pyyaml==6.0" \
    "rich==13.3.3" \
    "gputil==1.4.0" \
    "tensorboard==2.12.0" \
    "tensorboardx==2.6" \
    "tqdm==4.65.0"

print("\n[SUCCESS] All dependencies installed successfully.")
print("=" * 75)


## Section D: Dataset Setup (`bedroom_0` -> `scan0`)
Downloads the synthetic `bedroom_0.zip` dataset (~1.06 GB) from the official Kujiale CDN, extracts into `data/synthetic/scan0/`, and computes camera normalization parameters.

In [ ]:
# ==============================================================================
# SECTION D: DATASET SETUP (bedroom_0 / scan0)
# ==============================================================================
import os
import sys
import zipfile
import shutil

print("=" * 75)
print("             SECTION D: DATASET SETUP (bedroom_0 / scan0)")
print("=" * 75)

DATASET_URL = "https://kloudsim-usa-cos.kujiale.com/interiorverse/i2-sdf/i2-sdf/bedroom_0.zip"
DATA_DIR = os.path.join(".", "data", "synthetic")
SCAN0_DIR = os.path.join(DATA_DIR, "scan0")
ZIP_PATH = os.path.join(DATA_DIR, "bedroom_0.zip")

os.makedirs(DATA_DIR, exist_ok=True)

# Download and extract if scan0 is not already present
if not os.path.exists(SCAN0_DIR) or not os.path.exists(os.path.join(SCAN0_DIR, "cameras.npz")):
    if not os.path.exists(ZIP_PATH):
        print(f"[INFO] Downloading synthetic dataset (bedroom_0.zip, ~1.06 GB)...")
        print(f"       Source: {DATASET_URL}")
        !curl -C - -o "{ZIP_PATH}" "{DATASET_URL}"
    
    print(f"\n[INFO] Extracting {ZIP_PATH} ...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(DATA_DIR)
    
    extracted_candidates = [d for d in os.listdir(DATA_DIR) if d.startswith("scan") and d != "scan0"]
    if extracted_candidates:
        src_scan = os.path.join(DATA_DIR, extracted_candidates[0])
        print(f"[INFO] Mapping extracted '{extracted_candidates[0]}' -> 'scan0' ...")
        if os.path.exists(SCAN0_DIR):
            shutil.rmtree(SCAN0_DIR)
        shutil.move(src_scan, SCAN0_DIR)
    elif os.path.exists(os.path.join(DATA_DIR, "bedroom_0")):
        shutil.move(os.path.join(DATA_DIR, "bedroom_0"), SCAN0_DIR)

print(f"\n[STATUS] Target scan0 directory: {os.path.abspath(SCAN0_DIR)}")

# Generate cameras_normalize.npz if needed
cam_norm_path = os.path.join(SCAN0_DIR, "cameras_normalize.npz")
if not os.path.exists(cam_norm_path):
    print("[INFO] Generating cameras_normalize.npz using data/normalize_cameras.py ...")
    !python data/normalize_cameras.py --id 0 -n data/synthetic -r 2.0
else:
    print("[STATUS] cameras_normalize.npz already exists.")

# Verify dataset modalities
expected_subdirs = ["image", "depth", "normal", "hdr", "mask", "val"]
print("\n[VERIFICATION] Dataset contents in data/synthetic/scan0:")
for subdir in expected_subdirs:
    sub_path = os.path.join(SCAN0_DIR, subdir)
    if os.path.exists(sub_path):
        count = len(os.listdir(sub_path))
        print(f"  • {subdir + '/':<15} : {count} files found")
    else:
        print(f"  • {subdir + '/':<15} : [MISSING]")

print(f"  • {'cameras.npz':<15} : {'EXISTS' if os.path.exists(os.path.join(SCAN0_DIR, 'cameras.npz')) else 'MISSING'}")
print(f"  • {'cameras_normalize.npz':<15} : {'EXISTS' if os.path.exists(os.path.join(SCAN0_DIR, 'cameras_normalize.npz')) else 'MISSING'}")
print("=" * 75)


## Section E: Environment & Modality Verification
Verifies OpenEXR image decoding, PyTorch CUDA tensor allocation, and YAML configuration loading.

In [ ]:
# ==============================================================================
# SECTION E: ENVIRONMENT & MODALITY VERIFICATION
# ==============================================================================
import os
os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"
import cv2
import yaml
import torch
import numpy as np
import pytorch_lightning as pl
import utils

print("=" * 75)
print("             SECTION E: ENVIRONMENT & MODALITY VERIFICATION")
print("=" * 75)

# 1. Test OpenEXR image reading via OpenCV
depth_sample_path = os.path.join("data", "synthetic", "scan0", "depth", "0000.exr")
if os.path.exists(depth_sample_path):
    depth_img = cv2.imread(depth_sample_path, -1)
    if depth_img is not None:
        print(f"[SUCCESS] OpenEXR Decoding: Loaded {depth_sample_path} (Shape: {depth_img.shape}, Dtype: {depth_img.dtype})")
    else:
        print(f"[ERROR] Failed to decode OpenEXR image!")
else:
    print(f"[WARNING] Sample depth EXR not found at {depth_sample_path}")

# 2. Test CUDA Tensor Allocation
x = torch.randn(1000, 1000, device="cuda")
y = torch.matmul(x, x)
print(f"[SUCCESS] PyTorch CUDA Matrix Ops: Result Norm = {y.norm().item():.4f}")

# 3. Test Config Loading & Baseline Identity
for conf_name in ["config/synthetic.yml", "config/synthetic_physics.yml"]:
    with open(conf_name, 'r') as f:
        cfg = utils.CfgNode(yaml.load(f, Loader=yaml.FullLoader))
    g_weight = getattr(cfg.loss, 'ground_weight', 0.0)
    print(f"[SUCCESS] Config Loaded: {conf_name:<30} | ground_weight = {g_weight}")

print("=" * 75)
print("[STATUS] Environment and modalities verified successfully!")
print("=" * 75)


## Section F (Stage 1): Sanity Verification Test
> **Status**: `RUN_SANITY_TEST = True` by default.
> Executes fast unit tests (`tests/test_physics.py`) verifying floor heuristic estimation, probe generation, differentiable softmin loss, and strict gradient backpropagation to target MLP parameters.

In [ ]:
# ==============================================================================
# SECTION F (STAGE 1): SANITY VERIFICATION TEST
# ==============================================================================
# Fast unit test and math verification switch
RUN_SANITY_TEST = True

print("=" * 75)
print("            SECTION F (STAGE 1): SANITY VERIFICATION TEST")
print("=" * 75)
print(f"• Execution Switch: RUN_SANITY_TEST = {RUN_SANITY_TEST}")
print("=" * 75)

if RUN_SANITY_TEST:
    print("\n[1/2] Running Python syntax and compilation check...")
    !python -m py_compile model/physics.py model/network/__init__.py model/trainer/recon.py model/physics_classifier.py tests/test_physics.py
    print("[SUCCESS] py_compile passed with 0 errors.")
    
    print("\n[2/2] Running Unit Tests (tests/test_physics.py)...")
    !python -m unittest tests/test_physics.py
    print("\n[SUCCESS] Stage 1 Sanity Verification passed!")
else:
    print("[INFO] Sanity test skipped by user configuration.")
print("=" * 75)


## Section G (Stage 2): Controlled Runtime Calibration (Short Run Only)
> **Safety Guard**: `RUN_CALIBRATION = False` by default.
> **Important**: `MAX_STEPS = 1000` is used **strictly for measuring iterations-per-second, step latency, and GPU memory usage**.
> It is **NOT** presented as a final research reconstruction result. Set `RUN_CALIBRATION = True` only to profile runtime before starting full training.

In [ ]:
# ==============================================================================
# SECTION G (STAGE 2): RUNTIME CALIBRATION (SHORT RUN ONLY)
# ==============================================================================
# Calibration safety switch
RUN_CALIBRATION = False

# Short step count strictly for profiling latency and GPU throughput
MAX_STEPS = 1000  # Calibration only — NOT final research experiment

print("=" * 75)
print("            SECTION G (STAGE 2): RUNTIME CALIBRATION INSPECTION")
print("=" * 75)
print(f"• Calibration Step Count    : {MAX_STEPS} (Profiling only)")
print(f"• Execution Guard Status    : RUN_CALIBRATION = {RUN_CALIBRATION}")
print("=" * 75)

if not RUN_CALIBRATION:
    print("\n[ACTION REQUIRED] Runtime calibration was NOT launched.")
    print("To execute a short 1,000-step calibration run, set `RUN_CALIBRATION = True` and re-run this cell.")
else:
    print(f"\n[INFO] Launching 1,000-step Calibration Profiling...\n")
    !python main_recon.py --conf config/synthetic_physics.yml --scan_id 0 -d 0 --expname calibration_profile


## Section H (Stage 3): Full Research Experiment (Baseline & Physics)
> **Safety Guard**: `RUN_FULL_EXPERIMENT = False` by default.
> Set `RUN_FULL_EXPERIMENT = True` to launch the full 200,000-step training comparison:
> 1. **Baseline Model** (`config/synthetic.yml`, `ground_weight = 0.0`)
> 2. **Physics-Aware Model** (`config/synthetic_physics.yml`, `ground_weight = 0.1`)
>
> Training progress and losses are logged to TensorBoard.

In [ ]:
# ==============================================================================
# SECTION H (STAGE 3): FULL RESEARCH EXPERIMENT
# ==============================================================================
# Full experiment safety switch
RUN_FULL_EXPERIMENT = False

# Research experiment training budget
FULL_TRAIN_STEPS = 200000
SCAN_ID = 0
GPU_DEVICE_ID = 0

print("=" * 75)
print("          SECTION H (STAGE 3): FULL RESEARCH EXPERIMENT INSPECTION")
print("=" * 75)
print(f"• Target Training Budget    : {FULL_TRAIN_STEPS} steps")
print(f"• Target Scan ID            : {SCAN_ID}")
print(f"• Execution Guard Status    : RUN_FULL_EXPERIMENT = {RUN_FULL_EXPERIMENT}")
print("=" * 75)

if not RUN_FULL_EXPERIMENT:
    print("\n[ACTION REQUIRED] Full training experiment was NOT launched.")
    print("To execute full Baseline & Physics training, set `RUN_FULL_EXPERIMENT = True` above.")
    print("\nCommands that will be executed upon confirmation:")
    print(f"1. Baseline: python main_recon.py --conf config/synthetic.yml --scan_id {SCAN_ID} -d {GPU_DEVICE_ID} --expname synthetic_baseline")
    print(f"2. Physics : python main_recon.py --conf config/synthetic_physics.yml --scan_id {SCAN_ID} -d {GPU_DEVICE_ID} --expname synthetic_physics")
else:
    print(f"\n[1/2] Starting Full Baseline Training ({FULL_TRAIN_STEPS} steps)...\n")
    !python main_recon.py --conf config/synthetic.yml --scan_id {SCAN_ID} -d {GPU_DEVICE_ID} --expname synthetic_baseline
    
    print(f"\n[2/2] Starting Full Physics-Aware Training ({FULL_TRAIN_STEPS} steps)...\n")
    !python main_recon.py --conf config/synthetic_physics.yml --scan_id {SCAN_ID} -d {GPU_DEVICE_ID} --expname synthetic_physics
    print("\n[SUCCESS] Full research experiment training complete!")


## Section I (Stage 4): Ablation Studies
> **Safety Guard**: `RUN_ABLATIONS = False` by default.
> Evaluates the sensitivity of reconstruction to varying grounding loss weights $\lambda_g \in \{0.01, 0.05, 0.1, 0.5\}$.

In [ ]:
# ==============================================================================
# SECTION I (STAGE 4): ABLATION STUDIES
# ==============================================================================
# Ablation study safety switch
RUN_ABLATIONS = False
ABLATION_WEIGHTS = [0.01, 0.05, 0.1, 0.5]

print("=" * 75)
print("               SECTION I (STAGE 4): ABLATION STUDIES INSPECTION")
print("=" * 75)
print(f"• Grounding Weights to test : {ABLATION_WEIGHTS}")
print(f"• Execution Guard Status    : RUN_ABLATIONS = {RUN_ABLATIONS}")
print("=" * 75)

if not RUN_ABLATIONS:
    print("\n[ACTION REQUIRED] Ablation studies are disabled by default.")
    print("To execute grounding weight ablations, set `RUN_ABLATIONS = True` above and re-run.")
else:
    print("[INFO] Executing ablation experiments...")
    for w in ABLATION_WEIGHTS:
        exp = f"ablation_ground_{str(w).replace('.', '_')}"
        print(f"\n[RUNNING] Ablation weight = {w} (expname: {exp}) ...")
        !python main_recon.py --conf config/synthetic_physics.yml --scan_id 0 -d 0 --expname {exp}


## Section J (Stage 5): Mesh Extraction & Quantitative Evaluation
> **Safety Guard**: `RUN_MESH_EXTRACTION = False` by default.
> Extracts surface meshes at resolution 512 using Marching Cubes, calculates floor contact distribution metrics, and runs Random Forest physical feature analysis.

In [ ]:
# ==============================================================================
# SECTION J (STAGE 5): MESH EXTRACTION & QUANTITATIVE EVALUATION
# ==============================================================================
import os
import glob
import trimesh
import numpy as np

RUN_MESH_EXTRACTION = False
RESOLUTION = 512

print("=" * 75)
print("         SECTION J (STAGE 5): 3D MESH EXTRACTION & EVALUATION")
print("=" * 75)
print(f"• Marching Cubes Resolution : {RESOLUTION}")
print(f"• Execution Guard Status    : RUN_MESH_EXTRACTION = {RUN_MESH_EXTRACTION}")
print("=" * 75)

if RUN_MESH_EXTRACTION:
    print("\n[1/2] Extracting Baseline 3D Mesh...")
    !python main_recon.py --conf config/synthetic.yml --scan_id 0 -d 0 --expname synthetic_baseline --test --test_mode mesh --resolution {RESOLUTION}
    
    print("\n[2/2] Extracting Physics-Aware 3D Mesh...")
    !python main_recon.py --conf config/synthetic_physics.yml --scan_id 0 -d 0 --expname synthetic_physics --test --test_mode mesh --resolution {RESOLUTION}
    print("\n[SUCCESS] Mesh extraction complete!")
else:
    print("\n[INFO] Mesh extraction skipped (set `RUN_MESH_EXTRACTION = True` to extract).")

# Quantitative Evaluation Function
def evaluate_grounding_distribution(mesh_path, label="Model"):
    if not os.path.exists(mesh_path):
        return None
    m = trimesh.load(mesh_path)
    z = m.vertices[:, 2]
    return {
        'label': label,
        'vertices': len(m.vertices),
        'faces': len(m.faces),
        'min_z': float(np.min(z)),
        'p1_z': float(np.percentile(z, 1)),
        'p5_z': float(np.percentile(z, 5)),
        'mean_z': float(np.mean(z))
    }

b_meshes = glob.glob("exps/synthetic_baseline_0/**/*.ply", recursive=True)
p_meshes = glob.glob("exps/synthetic_physics_0/**/*.ply", recursive=True)

if b_meshes and p_meshes:
    b_eval = evaluate_grounding_distribution(b_meshes[0], "Baseline (Standard I²-SDF)")
    p_eval = evaluate_grounding_distribution(p_meshes[0], "Physics-Aware I²-SDF")
    print("\n" + "-" * 75)
    print(f"{'Metric':<30} | {'Baseline (Unconstrained)':<22} | {'Physics-Aware (Grounding)':<22}")
    print("-" * 75)
    print(f"{'Vertex Count':<30} | {b_eval['vertices']:<22} | {p_eval['vertices']:<22}")
    print(f"{'Face Count':<30} | {b_eval['faces']:<22} | {p_eval['faces']:<22}")
    print(f"{'Min Height (min z)':<30} | {b_eval['min_z']:<22.4f} | {p_eval['min_z']:<22.4f}")
    print(f"{'1st Percentile Height (p1)':<30} | {b_eval['p1_z']:<22.4f} | {p_eval['p1_z']:<22.4f}")
    print(f"{'5th Percentile Height (p5)':<30} | {b_eval['p5_z']:<22.4f} | {p_eval['p5_z']:<22.4f}")
    print("-" * 75)
else:
    print("\n[INFO] Checkpoint meshes will be evaluated here once training completes.")
print("=" * 75)


## Section K: Persistent Storage & Backup
> **Safety Guard**: `BACKUP_TO_GDRIVE = False` by default.
> Mounts Google Drive and copies all checkpoints, extracted meshes, and logs.

In [ ]:
# ==============================================================================
# SECTION K: PERSISTENT STORAGE (GOOGLE DRIVE BACKUP)
# ==============================================================================
import os
import shutil

BACKUP_TO_GDRIVE = False
GDRIVE_DEST_DIR = "/content/drive/MyDrive/i2sdf_experiments"

print("=" * 75)
print("             SECTION K: PERSISTENT STORAGE BACKUP")
print("=" * 75)
print(f"• Backup Destination       : {GDRIVE_DEST_DIR}")
print(f"• Execution Guard Status   : BACKUP_TO_GDRIVE = {BACKUP_TO_GDRIVE}")
print("=" * 75)

if not BACKUP_TO_GDRIVE:
    print("\n[INFO] Persistent backup is idle (set `BACKUP_TO_GDRIVE = True` to mount Drive and copy files).")
else:
    try:
        from google.colab import drive
        print("[INFO] Mounting Google Drive at /content/drive ...")
        drive.mount('/content/drive')
        os.makedirs(GDRIVE_DEST_DIR, exist_ok=True)
        if os.path.exists("exps"):
            print(f"[INFO] Copying exps -> {GDRIVE_DEST_DIR} ...")
            !cp -r exps/* "{GDRIVE_DEST_DIR}/"
            print(f"[SUCCESS] Backup complete to {GDRIVE_DEST_DIR}")
    except Exception as e:
        print(f"[ERROR] Backup failed: {e}")


## Summary & Exact Command Reference
Quick reference of CLI commands for all stages.

In [ ]:
# ==============================================================================
# SUMMARY: EXACT COMMAND CHEATSHEET
# ==============================================================================
print("=" * 80)
print("                I²-SDF REPRODUCIBILITY COMMAND CHEATSHEET")
print("=" * 80)
print("\n1. SANITY UNIT TESTS:")
print("   python -m unittest tests/test_physics.py")
print("\n2. CALIBRATION PROFILING (1,000 steps):")
print("   python main_recon.py --conf config/synthetic_physics.yml --scan_id 0 -d 0 --expname calibration_profile")
print("\n3. FULL BASELINE TRAINING (200,000 steps, ground_weight=0.0):")
print("   python main_recon.py --conf config/synthetic.yml --scan_id 0 -d 0 --expname synthetic_baseline")
print("\n4. FULL PHYSICS TRAINING (200,000 steps, ground_weight=0.1):")
print("   python main_recon.py --conf config/synthetic_physics.yml --scan_id 0 -d 0 --expname synthetic_physics")
print("\n5. BASELINE MESH EXTRACTION (Marching Cubes at Resolution 512):")
print("   python main_recon.py --conf config/synthetic.yml --scan_id 0 -d 0 --expname synthetic_baseline --test --test_mode mesh --resolution 512")
print("\n6. PHYSICS MESH EXTRACTION (Marching Cubes at Resolution 512):")
print("   python main_recon.py --conf config/synthetic_physics.yml --scan_id 0 -d 0 --expname synthetic_physics --test --test_mode mesh --resolution 512")
print("=" * 80)
